In [ ]:
from google.colab import drive

# Drive'a erişim izni
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!unzip /content/drive/MyDrive/kaynak_klasor.zip

Archive:  /content/drive/MyDrive/kaynak_klasor.zip
  inflating: kaynak_klasor/autorun.inf  
 extracting: kaynak_klasor/CDViewer.zip  
   creating: kaynak_klasor/ComponentUpdate/
  inflating: kaynak_klasor/ComponentUpdate/ComponentUpdate.exe  
  inflating: kaynak_klasor/ComponentUpdate/readme_american.txt  
   creating: kaynak_klasor/DICOM/
   creating: kaynak_klasor/DICOM/0000ECDC/
   creating: kaynak_klasor/DICOM/0000ECDC/AA36FA59/
   creating: kaynak_klasor/DICOM/0000ECDC/AA36FA59/AA663247/
   creating: kaynak_klasor/DICOM/0000ECDC/AA36FA59/AA663247/000018CD/
  inflating: kaynak_klasor/DICOM/0000ECDC/AA36FA59/AA663247/000018CD/EE0234A4  
  inflating: kaynak_klasor/DICOM/0000ECDC/AA36FA59/AA663247/000018CD/EE02F3C3  
  inflating: kaynak_klasor/DICOM/0000ECDC/AA36FA59/AA663247/000018CD/EE034D87  
  inflating: kaynak_klasor/DICOM/0000ECDC/AA36FA59/AA663247/000018CD/EE063DF5  
  inflating: kaynak_klasor/DICOM/0000ECDC/AA36FA59/AA663247/000018CD/EE07E0DC  
  inflating: kaynak_klasor/DICOM

In [ ]:
import shutil

def move_folder(source_folder, destination_folder):
    try:
        # Klasörü yeni konuma taşı
        shutil.move(source_folder, destination_folder)
        print(f"Klasör başarıyla taşındı: {source_folder} -> {destination_folder}")
    except OSError as e:
        print(f"Hata: Klasör taşınırken bir sorun oluştu: {e.strerror}")

# Klasörü taşımak için kullanılacak kaynak ve hedef yollar
source_folder = '/content/kaynak_klasor'
destination_folder = '/content/drive/MyDrive'

# Klasörü taşıma işlemi
move_folder(source_folder, destination_folder)



Klasör başarıyla taşındı: /content/kaynak_klasor -> /content/drive/MyDrive


In [ ]:
!pip install pydicom

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 14.1 MB/s eta 0:00:00


In [ ]:
import os
import shutil
import pydicom
from PIL import Image

def clean_and_move_dicom_files(source_folder):
    items = os.listdir(source_folder)
    for item in items:
        item_path = os.path.join(source_folder, item)
        if item != "DICOM" and os.path.isdir(item_path):
            shutil.rmtree(item_path)
        elif item != "DICOM" and os.path.isfile(item_path):
            os.remove(item_path)

    for root, dirs, files in os.walk(os.path.join(source_folder, "DICOM"), topdown=False):
        if not dirs:
            for file in files:
                shutil.move(os.path.join(root, file), source_folder)
            break

    shutil.rmtree(os.path.join(source_folder, "DICOM"))

def dicom_to_png_filtered(source_folder, destination_folder, start_index, end_index):
    clean_and_move_dicom_files(source_folder)

    files = [os.path.join(source_folder, f) for f in os.listdir(source_folder)]

    def sort_key(filepath):
        dicom_image = pydicom.dcmread(filepath)
        return int(dicom_image.get('InstanceNumber', 9999))

    files_sorted = sorted(files, key=sort_key)

    os.makedirs(destination_folder, exist_ok=True)

    for i, file_path in enumerate(files_sorted, start=1):
        if start_index <= i <= end_index:
            dicom_image = pydicom.dcmread(file_path)
            image = Image.fromarray(dicom_image.pixel_array)

            # Orijinal dosya ismini koruyarak PNG olarak kaydet
            original_file_name = os.path.splitext(os.path.basename(file_path))[0] + ".png"
            image.save(os.path.join(destination_folder, original_file_name), "PNG")
        else:
            os.remove(file_path)

    print(f"{start_index} ile {end_index} arasındaki DICOM görüntüleri dönüştürüldü ve diğerleri silindi.")

# Kullanıcıdan aralık değerlerini alın ve işlemi başlatın
start_index = int(input("Başlangıç index'i giriniz: "))
end_index = int(input("Bitiş index'i giriniz: "))

source_folder = "/content/drive/MyDrive/kaynak_klasor"  # DICOM dosyalarının taşınacağı ve işleneceği klasör yolu
destination_folder = "/content/drive/MyDrive/hedef_klasor"  # Filtrelenmiş PNG dosyalarının kaydedileceği klasör yolu

dicom_to_png_filtered(source_folder, destination_folder, start_index, end_index)


Başlangıç index'i giriniz: 118
Bitiş index'i giriniz: 199
118 ile 199 arasındaki DICOM görüntüleri dönüştürüldü ve diğerleri silindi.


In [ ]:
len(os.listdir("/content/drive/MyDrive/hedef_klasor"))

82

In [ ]:
import os
import numpy as np
from keras.models import load_model
from PIL import Image

def predict_and_move_images(model_path, source_folder, target_normal_folder, target_abnormal_folder):
    # Modeli yükle
    model = load_model(model_path)

    # Kaynak klasördeki PNG dosyalarını al
    image_files = [f for f in os.listdir(source_folder) if f.endswith('.png')]

    # Her bir görüntü için tahmin yap ve uygun klasöre taşı
    for image_file in image_files:
        image_path = os.path.join(source_folder, image_file)

        # Görüntüyü yükle ve model ile tahmin yap
        img = Image.open(image_path)
        img = img.resize((256, 256))  # Model giriş boyutuna uygun şekilde yeniden boyutlandır
        img_array = np.expand_dims(np.array(img) / 255.0, axis=0)  # Normalleştirme ve boyut değiştirme
        prediction = model.predict(img_array)

        # Tahmini değerlendir ve uygun klasöre taşı
        if np.argmax(prediction) == 0:  # Tahmin [1, 0] ise abnormal
            target_folder = target_abnormal_folder
        else:  # Tahmin [0, 1] ise normal
            target_folder = target_normal_folder

        # Hedef klasör varsa kontrol et, yoksa oluştur
        os.makedirs(target_folder, exist_ok=True)

        # Dosyayı taşı
        destination_path = os.path.join(target_folder, image_file)
        os.replace(image_path, destination_path)

        print(f"{image_file} dosyası {target_folder} klasörüne taşındı.")

# Kullanım
model_path = "/content/drive/MyDrive/model_in_5_1.h5"  # Modelin yolunu belirtin
source_folder = "/content/drive/MyDrive/hedef_klasor"  # Tahmin yapılacak görüntülerin klasörü
target_normal_folder = "/content/drive/MyDrive/normal"  # Normal görüntülerin taşınacağı klasör
target_abnormal_folder = "/content/drive/MyDrive/abnormal"  # Anormal görüntülerin taşınacağı klasör

predict_and_move_images(model_path, source_folder, target_normal_folder, target_abnormal_folder)


1/1 [==============================] - 4s 4s/step
EEF8382D.png dosyası /content/drive/MyDrive/normal klasörüne taşındı.
1/1 [==============================] - 0s 27ms/step
EE0E0897.png dosyası /content/drive/MyDrive/normal klasörüne taşındı.
1/1 [==============================] - 0s 25ms/step
EEC7E3D8.png dosyası /content/drive/MyDrive/normal klasörüne taşındı.
1/1 [==============================] - 0s 27ms/step
EEE70BB4.png dosyası /content/drive/MyDrive/normal klasörüne taşındı.
1/1 [==============================] - 0s 25ms/step
EE1A608F.png dosyası /content/drive/MyDrive/normal klasörüne taşındı.
1/1 [==============================] - 0s 26ms/step
EE15813B.png dosyası /content/drive/MyDrive/normal klasörüne taşındı.
1/1 [==============================] - 0s 27ms/step
EEA7F61B.png dosyası /content/drive/MyDrive/normal klasörüne taşındı.
1/1 [==============================] - 0s 26ms/step
EEC46C7B.png dosyası /content/drive/MyDrive/normal klasörüne taşındı.
1/1 [=====================

In [ ]:
len(os.listdir("/content/drive/MyDrive/abnormal"))

51

In [ ]:
pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 719.5/719.5 kB 13.5 MB/s eta 0:00:00


In [ ]:
import os
import cv2
import numpy as np
from ultralytics import YOLO  # YOLO kütüphanesini doğru bir şekilde yüklediğinizden emin olun

# Model yüklemesi
model = YOLO('/content/drive/MyDrive/ImageSegmentationYOLOv8/runs/train/exp/weights/best.pt')

# Segmentasyon sonuçlarını kaydetme klasörü
segmented_dir = '/content/drive/MyDrive/segmented'
os.makedirs(segmented_dir, exist_ok=True)  # Klasör yoksa oluştur

# Tahminler için klasör yolu
predict_images_dir = '/content/drive/MyDrive/abnormal'  # Tahmin yapılacak görüntülerin yolu

# Her bir görüntü üzerinde işlem yap
for image_file in os.listdir(predict_images_dir):
    if image_file.endswith('.png'):
        img_path = os.path.join(predict_images_dir, image_file)
        img = cv2.imread(img_path)
        H, W, _ = img.shape

        # Model ile tahmin yap
        results = model(img)

        # Tahmin edilen maskeleri görselleştir
        if results[0].masks is not None:
            predicted_masks = results[0].masks.data.cpu().numpy()
            for mask in predicted_masks:
                mask = cv2.resize(mask, (W, H))  # Maskeyi orijinal görüntü boyutlarına yeniden boyutlandır
                _, mask_thresh = cv2.threshold(mask, 0.5, 255, cv2.THRESH_BINARY)
                mask_thresh = mask_thresh.astype(np.uint8)
                contours, _ = cv2.findContours(mask_thresh, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
                cv2.drawContours(img, contours, -1, (0, 255, 0), 0)  # Konturları yeşil renk ile çiz

        # Segmentasyon sonucunu kaydet
        save_path = os.path.join(segmented_dir, image_file)
        cv2.imwrite(save_path, img)

print("Tüm segmentasyon sonuçları 'segmented' klasörüne kaydedildi.")



0: 640x640 1 Cancer, 11.5ms
Speed: 17.0ms preprocess, 11.5ms inference, 756.2ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 1 Cancer, 11.5ms
Speed: 3.9ms preprocess, 11.5ms inference, 1.9ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 2 Cancers, 11.5ms
Speed: 3.9ms preprocess, 11.5ms inference, 2.2ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 3 Cancers, 11.5ms
Speed: 3.2ms preprocess, 11.5ms inference, 30.6ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 4 Cancers, 11.4ms
Speed: 2.5ms preprocess, 11.4ms inference, 2.2ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 3 Cancers, 11.9ms
Speed: 2.5ms preprocess, 11.9ms inference, 2.0ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 2 Cancers, 11.5ms
Speed: 2.7ms preprocess, 11.5ms inference, 1.9ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 2 Cancers, 11.4ms
Speed: 2.8ms preprocess, 11.4ms inference, 2.0ms postprocess per image at shape

In [ ]:
import shutil
import os

# Kaynak klasör yolları
normal_dir = '/content/drive/MyDrive/normal'
segmented_dir = '/content/drive/MyDrive/segmented'

# Hedef klasör yolu
output_dir = '/content/drive/MyDrive/output'
os.makedirs(output_dir, exist_ok=True)  # Eğer klasör yoksa oluştur

# 'normal' klasöründeki görüntüleri kopyala
for filename in os.listdir(normal_dir):
    if filename.endswith('.png'):
        src_path = os.path.join(normal_dir, filename)
        dst_path = os.path.join(output_dir, filename)
        shutil.copy(src_path, dst_path)

# 'segmented' klasöründeki görüntüleri kopyala
for filename in os.listdir(segmented_dir):
    if filename.endswith('.png'):
        src_path = os.path.join(segmented_dir, filename)
        dst_path = os.path.join(output_dir, filename)
        shutil.copy(src_path, dst_path)

print("Tüm görüntüler 'output' klasörüne kopyalandı.")


Tüm görüntüler 'output' klasörüne kopyalandı.


In [ ]:
os.makedirs('/content/drive/MyDrive/hasta_sonuclari', exist_ok=True)

In [ ]:
import os
import copy  # Eksik olan modülü içe aktar
import numpy as np
import pydicom
from PIL import Image
from pydicom.dataset import Dataset, FileMetaDataset

def is_color_dicom(dicom_path):
    # DICOM dosyasını oku
    ds = pydicom.dcmread(dicom_path)
    # Photometric Interpretation değerine bakarak renkli olup olmadığını kontrol et
    return ds.PhotometricInterpretation.startswith('RGB')

def convert_to_color(png_image):
    # Siyah beyaz (1 kanal) PNG görüntüyü renkli (3 kanal) bir görüntüye çevir
    return png_image.convert('RGB')

def resize_image(png_image, width, height):
    # Görüntüyü istenilen boyutlara uyacak şekilde yeniden boyutlandır
    return png_image.resize((width, height))

def create_dicom_from_png(png_image, original_dicom, instance_number):
    # Orijinal DICOM verilerini kopyala
    new_dicom = copy.deepcopy(original_dicom)

    # PNG görüntüsünü Numpy dizisine dönüştür ve PixelData olarak ayarla
    new_dicom.PixelData = np.array(png_image).tobytes()

    # Görüntünün boyutlarını güncelle
    new_dicom.Rows, new_dicom.Columns = png_image.size

    # Eğer orijinal DICOM renkli değilse, Photometric Interpretation'ı güncelle
    if not is_color_dicom(original_dicom.filename):
        new_dicom.PhotometricInterpretation = 'RGB'

    # Örneklerin piksel başına sayısını güncelle
    new_dicom.SamplesPerPixel = 3
    new_dicom.BitsAllocated = 8
    new_dicom.BitsStored = 8
    new_dicom.HighBit = 7
    new_dicom.PixelRepresentation = 0

    # Transfer Syntax ve diğer meta verileri ayarla
    new_dicom.file_meta.TransferSyntaxUID = pydicom.uid.ExplicitVRLittleEndian

    # Instance Number'ı güncelle
    new_dicom.InstanceNumber = instance_number

    return new_dicom


def convert_png_to_dicom(png_folder, original_dicom_folder, output_folder):
    # PNG ve DICOM dosya yollarını kontrol et
    instance_numbers = {}  # Sıralama numarasını tutmak için sözlük

    # Dosyaların instance numarasını al ve bir listeye kaydet
    for dicom_filename in os.listdir(original_dicom_folder):
        dicom_path = os.path.join(original_dicom_folder, dicom_filename)
        original_dicom = pydicom.dcmread(dicom_path)
        instance_number = original_dicom.get('InstanceNumber', 0)
        instance_numbers[dicom_filename] = instance_number

    # Sıralama numarasına göre PNG dosyalarını sırala
    sorted_png_filenames = sorted(os.listdir(png_folder), key=lambda fname: instance_numbers.get(fname.replace('.png', ''), 0))

    # Sıralanmış dosyalar üzerinde işlem yap
    for i, png_filename in enumerate(sorted_png_filenames, start=1):
        if png_filename.lower().endswith('.png'):
            # Yolları tanımla
            png_path = os.path.join(png_folder, png_filename)
            dicom_filename_without_ext = png_filename.split('.')[0]
            original_dicom_path = os.path.join(original_dicom_folder, dicom_filename_without_ext)
            output_dicom_path = os.path.join(output_folder, dicom_filename_without_ext )

            # Orijinal DICOM dosyasını yükle
            original_dicom = pydicom.dcmread(original_dicom_path)

            # PNG görüntüsünü yükle
            png_image = Image.open(png_path).convert('RGB')

            # Renk ve boyut ayarlamalarını yap
            png_image = resize_image(png_image, 512, 512)

            # Yeni DICOM oluştur ve InstanceNumber ayarla
            new_dicom = create_dicom_from_png(png_image, original_dicom, i)

            # Yeni DICOM dosyasını kaydet
            new_dicom.save_as(output_dicom_path)

            print(f"Converted {png_filename} to DICOM and saved as {output_dicom_path} with InstanceNumber {i}")




# Kullanım
png_folder =  '/content/drive/MyDrive/output'
original_dicom_folder = '/content/drive/MyDrive/kaynak_klasor'
output_folder = '/content/drive/MyDrive/hasta_sonuclari'
convert_png_to_dicom(png_folder, original_dicom_folder, output_folder)


Converted EEF8382D.png to DICOM and saved as /content/drive/MyDrive/hasta_sonuclari/EEF8382D with InstanceNumber 1
Converted EE0E0897.png to DICOM and saved as /content/drive/MyDrive/hasta_sonuclari/EE0E0897 with InstanceNumber 2
Converted EEC7E3D8.png to DICOM and saved as /content/drive/MyDrive/hasta_sonuclari/EEC7E3D8 with InstanceNumber 3
Converted EEE70BB4.png to DICOM and saved as /content/drive/MyDrive/hasta_sonuclari/EEE70BB4 with InstanceNumber 4
Converted EE1A608F.png to DICOM and saved as /content/drive/MyDrive/hasta_sonuclari/EE1A608F with InstanceNumber 5
Converted EE15813B.png to DICOM and saved as /content/drive/MyDrive/hasta_sonuclari/EE15813B with InstanceNumber 6
Converted EEA7F61B.png to DICOM and saved as /content/drive/MyDrive/hasta_sonuclari/EEA7F61B with InstanceNumber 7
Converted EEC46C7B.png to DICOM and saved as /content/drive/MyDrive/hasta_sonuclari/EEC46C7B with InstanceNumber 8
Converted EEC7A3C5.png to DICOM and saved as /content/drive/MyDrive/hasta_sonucl

In [ ]:
len(os.listdir("/content/drive/MyDrive/hasta_sonuclari"))

82

In [ ]:
import matplotlib.pyplot as plt
import pydicom
import os

def load_sorted_dicoms(dicom_folder):
    dicom_files = [os.path.join(dicom_folder, f) for f in sorted(os.listdir(dicom_folder))]
    dicoms = [pydicom.dcmread(d_file) for d_file in dicom_files]
    # Dosyaları 'InstanceNumber' özelliğine göre sırala
    dicoms.sort(key=lambda x: x.InstanceNumber)
    return dicoms

def show_dicom_images(dicoms):
    # DICOM görüntülerini sıralı şekilde göster
    fig, ax = plt.subplots()
    plt.ion()  # Etkileşimli modu etkinleştir

    for dicom in dicoms:
        img = dicom.pixel_array
        ax.imshow(img, cmap='gray')  # Gri tonlamalı olarak ayarla, RGB için cmap parametresini kaldırın
        ax.set_title(f"Instance Number: {dicom.InstanceNumber}")
        plt.draw()
        plt.pause(0.5)  # Görüntüler arasında yarım saniye bekle

    plt.ioff()  # Etkileşimli modu devre dışı bırak
    plt.show()

# Kullanım
dicom_folder = '/content/drive/MyDrive/hasta_sonuclari'
dicoms = load_sorted_dicoms(dicom_folder)
show_dicom_images(dicoms)


interactive(children=(IntSlider(value=0, description='index', max=81), Output()), _dom_classes=('widget-intera…

In [ ]:
"""
import shutil

# Klasörleri silme işlevi
def delete_folder(folder_path):
    try:
        shutil.rmtree(folder_path)
        print(f"Klasör başarıyla silindi: {folder_path}")
    except OSError as e:
        print(f"Hata: {folder_path} klasörü silinirken bir sorun oluştu: {e.strerror}")

# Silinecek klasör yolları
folders_to_delete = [
    '/content/drive/MyDrive/abnormal',
    '/content/drive/MyDrive/hasta_sonuclari',
    '/content/drive/MyDrive/hedef_klasor',
    '/content/drive/MyDrive/kaynak_klasor',
    '/content/drive/MyDrive/output',
    '/content/drive/MyDrive/normal',
    '/content/drive/MyDrive/segmented'


]

# Her bir klasörü sil
for folder in folders_to_delete:
    delete_folder(folder)
"""

'\nimport shutil\n\n# Klasörleri silme işlevi\ndef delete_folder(folder_path):\n    try:\n        shutil.rmtree(folder_path)\n        print(f"Klasör başarıyla silindi: {folder_path}")\n    except OSError as e:\n        print(f"Hata: {folder_path} klasörü silinirken bir sorun oluştu: {e.strerror}")\n\n# Silinecek klasör yolları\nfolders_to_delete = [\n    \'/content/drive/MyDrive/abnormal\',\n    \'/content/drive/MyDrive/hasta_sonuclari\',\n    \'/content/drive/MyDrive/hedef_klasor\',\n    \'/content/drive/MyDrive/kaynak_klasor\',\n    \'/content/drive/MyDrive/output\',\n    \'/content/drive/MyDrive/normal\',\n    \'/content/drive/MyDrive/segmented\'\n    \n\n]\n\n# Her bir klasörü sil\nfor folder in folders_to_delete:\n    delete_folder(folder)\n'